# ORCID Works Query

Created by [Matt Artz](https://www.mattartz.me/) | [GitHub](https://github.com/MattArtzAnthro) | [ORCID](https://orcid.org/0000-0002-3822-1429)

---

## What This Notebook Does

This notebook queries the ORCID Public API to retrieve complete publication/works data from researcher profiles. Unlike CrossRef (which only knows about works where publishers registered the ORCID), this queries the author's own ORCID profile to get all works they have claimed or added—including books, chapters, datasets, conference papers, and works from publishers who don't register ORCIDs.

The ORCID works section contains rich metadata including titles, DOIs, other external identifiers (PubMed, arXiv, ISBN, etc.), publication dates, journal/venue information, contributor lists, and source information showing where each work entry originated (CrossRef, Scopus, manually added, etc.).

## Key Features

- **Complete Works Extraction**: Retrieves all works from ORCID profile, not just CrossRef-linked
- **Multiple Identifier Support**: Extracts DOIs, PubMed IDs, arXiv IDs, ISBNs, handles, and more
- **Source Tracking**: Shows where each work entry came from (CrossRef, Scopus, DataCite, manual)
- **Contributor Parsing**: Extracts co-author information with roles and ORCIDs when available
- **Work Type Classification**: Distinguishes journal articles, books, chapters, datasets, etc.
- **Batch Processing**: Query multiple ORCIDs in sequence with rate limiting

## Workflow

1. **Setup**: Install dependencies and configure ORCID API
2. **Input**: Enter single ORCID or upload list of ORCIDs
3. **Query**: Retrieve works summaries, then fetch full details for each work
4. **Parse**: Extract all metadata fields into structured format
5. **Export**: Download CSV and JSON with complete works data

## Comparison with CrossRef Query

| Aspect | ORCID Works Query (this notebook) | CrossRef Author Query |
|--------|-----------------------------------|----------------------|
| Data Source | Author's ORCID profile | CrossRef metadata registry |
| Coverage | All works author has claimed | Only works with registered ORCID |
| Work Types | Articles, books, datasets, software, etc. | Primarily journal articles |
| Abstracts | Usually not available | Often available |
| References | Not available | Available |
| Citation Counts | Not available | Available |

## Citation

> Artz, M. (2026). Wikidata Tools. GitHub. https://github.com/MattArtzAnthro/wikidata-tools

*A citable DOI will be available via Zenodo.*

## License

[CC BY-NC 4.0](https://creativecommons.org/licenses/by-nc/4.0/)

## Setup

In [ ]:
# Install required packages
!pip install requests pandas ipywidgets -q

import requests
import pandas as pd
import json
import re
import time
import os
from datetime import datetime
from typing import Dict, List, Optional, Tuple, Any
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets
from io import StringIO

# Google Colab file handling
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("✅ Setup complete.")
print(f"   Running in Colab: {IN_COLAB}")

## Configuration

In [ ]:
# ORCID API Configuration
class Config:
    # ORCID Public API v3.0
    ORCID_API_BASE = "https://pub.orcid.org/v3.0"
    
    # Rate limiting (Public API allows ~24 req/sec, but we're conservative)
    REQUEST_DELAY = 0.5  # seconds between requests
    TIMEOUT = 30  # seconds
    MAX_RETRIES = 3
    
    # User-Agent for polite API access
    USER_AGENT = "ORCID-Works-Query/1.0 (Anthropology Wikidata Project; mailto:matt@mattartz.me)"
    
    # Output settings
    OUTPUT_PATH = "/content/outputs/" if IN_COLAB else "outputs/"
    
    # Whether to fetch full work details (slower but more complete)
    FETCH_FULL_DETAILS = True

config = Config()

# Create output directory
os.makedirs(config.OUTPUT_PATH, exist_ok=True)

print("⚙️ Configuration")
print("=" * 50)
print(f"ORCID API: {config.ORCID_API_BASE}")
print(f"Request delay: {config.REQUEST_DELAY}s")
print(f"Fetch full details: {config.FETCH_FULL_DETAILS}")
print(f"Output path: {config.OUTPUT_PATH}")

## ORCID Validation Functions

In [ ]:
def validate_orcid_checksum(orcid: str) -> bool:
    """
    Validate ORCID checksum using ISO 7064 Mod 11-2 algorithm.
    """
    digits = orcid.replace('-', '')
    if len(digits) != 16:
        return False
    
    total = 0
    for char in digits[:-1]:
        if not char.isdigit():
            return False
        total = (total + int(char)) * 2
    
    remainder = total % 11
    result = (12 - remainder) % 11
    check_digit = 'X' if result == 10 else str(result)
    
    return digits[-1].upper() == check_digit


def clean_orcid(orcid_input: str) -> Optional[str]:
    """
    Extract and normalize ORCID from various input formats.
    Returns formatted ORCID (0000-0000-0000-000X) or None if invalid.
    """
    if not orcid_input or not isinstance(orcid_input, str):
        return None
    
    value = str(orcid_input).strip()
    
    # Patterns to match ORCID
    patterns = [
        r'(?:https?://)?(?:www\.)?orcid\.org/(\d{4}-?\d{4}-?\d{4}-?\d{3}[\dXx])',
        r'orcid[:/]?(\d{4}-?\d{4}-?\d{4}-?\d{3}[\dXx])',
        r'\b(\d{4}-\d{4}-\d{4}-\d{3}[\dXx])\b',
        r'\b(\d{16})\b',
        r'\b(\d{15}[Xx])\b'
    ]
    
    for pattern in patterns:
        match = re.search(pattern, value, re.IGNORECASE)
        if match:
            orcid = match.group(1).replace('-', '').upper()
            formatted = f"{orcid[:4]}-{orcid[4:8]}-{orcid[8:12]}-{orcid[12:16]}"
            
            if validate_orcid_checksum(formatted):
                return formatted
    
    return None


print("✅ Validation functions loaded.")

# Test validation
test_orcids = [
    "0000-0002-1825-0097",  # Valid
    "https://orcid.org/0000-0002-1825-0097",  # URL format
    "0000-0002-1825-0098",  # Invalid checksum
]
print("\nValidation tests:")
for test in test_orcids:
    result = clean_orcid(test)
    print(f"  {test[:40]:40s} → {result or 'INVALID'}")

## ORCID API Functions

In [ ]:
def get_orcid_headers() -> Dict[str, str]:
    """Get headers for ORCID API requests."""
    return {
        'Accept': 'application/json',
        'User-Agent': config.USER_AGENT
    }


def fetch_orcid_works_summary(orcid: str) -> Tuple[Optional[List[Dict]], Optional[Dict]]:
    """
    Fetch works summary from ORCID profile.
    
    Returns:
        (list of work groups, person info dict) or (None, None) on error
    """
    url = f"{config.ORCID_API_BASE}/{orcid}/works"
    
    try:
        response = requests.get(
            url,
            headers=get_orcid_headers(),
            timeout=config.TIMEOUT
        )
        
        if response.status_code == 404:
            print(f"  ORCID {orcid} not found")
            return None, None
        
        response.raise_for_status()
        data = response.json()
        
        # Extract work groups
        groups = data.get('group', [])
        
        # Get person info for context
        person_info = {
            'last_modified': data.get('last-modified-date', {}).get('value'),
            'path': data.get('path')
        }
        
        return groups, person_info
        
    except requests.exceptions.RequestException as e:
        print(f"  Error fetching works for {orcid}: {e}")
        return None, None


def fetch_work_details(orcid: str, put_code: str) -> Optional[Dict]:
    """
    Fetch full details for a specific work.
    
    Args:
        orcid: ORCID identifier
        put_code: Work's put-code identifier
    
    Returns:
        Full work record or None on error
    """
    url = f"{config.ORCID_API_BASE}/{orcid}/work/{put_code}"
    
    try:
        response = requests.get(
            url,
            headers=get_orcid_headers(),
            timeout=config.TIMEOUT
        )
        
        if response.status_code == 404:
            return None
        
        response.raise_for_status()
        return response.json()
        
    except requests.exceptions.RequestException as e:
        print(f"    Error fetching work {put_code}: {e}")
        return None


def fetch_person_info(orcid: str) -> Optional[Dict]:
    """
    Fetch basic person info from ORCID.
    """
    url = f"{config.ORCID_API_BASE}/{orcid}/person"
    
    try:
        response = requests.get(
            url,
            headers=get_orcid_headers(),
            timeout=config.TIMEOUT
        )
        
        if response.status_code == 404:
            return None
        
        response.raise_for_status()
        data = response.json()
        
        name_data = data.get('name', {}) or {}
        
        return {
            'given_name': name_data.get('given-names', {}).get('value', '') if name_data.get('given-names') else '',
            'family_name': name_data.get('family-name', {}).get('value', '') if name_data.get('family-name') else '',
            'credit_name': name_data.get('credit-name', {}).get('value', '') if name_data.get('credit-name') else ''
        }
        
    except requests.exceptions.RequestException:
        return None


print("✅ ORCID API functions loaded.")

## Work Parsing Functions

In [ ]:
def parse_date(date_obj: Optional[Dict]) -> Tuple[Optional[str], Optional[int], Optional[int], Optional[int]]:
    """
    Parse ORCID date object into components.
    
    Returns:
        (iso_date, year, month, day)
    """
    if not date_obj:
        return None, None, None, None
    
    year = date_obj.get('year', {}).get('value') if date_obj.get('year') else None
    month = date_obj.get('month', {}).get('value') if date_obj.get('month') else None
    day = date_obj.get('day', {}).get('value') if date_obj.get('day') else None
    
    # Convert to integers
    year = int(year) if year else None
    month = int(month) if month else None
    day = int(day) if day else None
    
    # Build ISO date
    if year:
        if month and day:
            iso_date = f"{year:04d}-{month:02d}-{day:02d}"
        elif month:
            iso_date = f"{year:04d}-{month:02d}"
        else:
            iso_date = f"{year:04d}"
    else:
        iso_date = None
    
    return iso_date, year, month, day


def parse_external_ids(external_ids_obj: Optional[Dict]) -> Dict[str, List[str]]:
    """
    Parse external identifiers from work.
    
    Returns:
        Dict mapping identifier type to list of values
    """
    if not external_ids_obj:
        return {}
    
    ids = {}
    for ext_id in external_ids_obj.get('external-id', []):
        id_type = ext_id.get('external-id-type', '').lower()
        id_value = ext_id.get('external-id-value', '')
        id_url = ext_id.get('external-id-url', {}).get('value', '') if ext_id.get('external-id-url') else ''
        id_relationship = ext_id.get('external-id-relationship', '')
        
        if id_type and id_value:
            if id_type not in ids:
                ids[id_type] = []
            ids[id_type].append({
                'value': id_value,
                'url': id_url,
                'relationship': id_relationship
            })
    
    return ids


def parse_contributors(contributors_obj: Optional[Dict]) -> List[Dict]:
    """
    Parse contributors (co-authors) from work.
    """
    if not contributors_obj:
        return []
    
    contributors = []
    for contrib in contributors_obj.get('contributor', []):
        name_obj = contrib.get('credit-name', {})
        orcid_obj = contrib.get('contributor-orcid', {})
        attrs = contrib.get('contributor-attributes', {}) or {}
        
        contributor = {
            'name': name_obj.get('value', '') if name_obj else '',
            'orcid': orcid_obj.get('path', '') if orcid_obj else '',
            'orcid_uri': orcid_obj.get('uri', '') if orcid_obj else '',
            'role': attrs.get('contributor-role', ''),
            'sequence': attrs.get('contributor-sequence', '')
        }
        
        if contributor['name'] or contributor['orcid']:
            contributors.append(contributor)
    
    return contributors


def parse_work_summary(work_summary: Dict) -> Dict:
    """
    Parse a work summary from the works endpoint.
    """
    # Title
    title_obj = work_summary.get('title', {}) or {}
    title = title_obj.get('title', {}).get('value', '') if title_obj.get('title') else ''
    subtitle = title_obj.get('subtitle', {}).get('value', '') if title_obj.get('subtitle') else ''
    translated_title = title_obj.get('translated-title', {}).get('value', '') if title_obj.get('translated-title') else ''
    
    # Journal/venue
    journal_title = work_summary.get('journal-title', {}).get('value', '') if work_summary.get('journal-title') else ''
    
    # Date
    pub_date, pub_year, pub_month, pub_day = parse_date(work_summary.get('publication-date'))
    
    # External IDs
    external_ids = parse_external_ids(work_summary.get('external-ids'))
    
    # Extract primary DOI
    doi = ''
    doi_ids = external_ids.get('doi', [])
    if doi_ids:
        # Prefer 'self' relationship
        for d in doi_ids:
            if d.get('relationship') == 'self':
                doi = d.get('value', '')
                break
        if not doi:
            doi = doi_ids[0].get('value', '')
    
    # Source
    source_obj = work_summary.get('source', {}) or {}
    source_name = ''
    if source_obj.get('source-name'):
        source_name = source_obj['source-name'].get('value', '')
    elif source_obj.get('source-client-id'):
        source_name = source_obj['source-client-id'].get('path', '')
    
    return {
        'put_code': work_summary.get('put-code'),
        'title': title,
        'subtitle': subtitle,
        'translated_title': translated_title,
        'journal_title': journal_title,
        'type': work_summary.get('type', ''),
        'publication_date': pub_date,
        'publication_year': pub_year,
        'publication_month': pub_month,
        'publication_day': pub_day,
        'doi': doi,
        'external_ids': external_ids,
        'source': source_name,
        'visibility': work_summary.get('visibility', ''),
        'path': work_summary.get('path', ''),
        'display_index': work_summary.get('display-index', '')
    }


def parse_work_full(work_data: Dict) -> Dict:
    """
    Parse full work details from individual work endpoint.
    """
    # Start with summary fields
    parsed = parse_work_summary(work_data)
    
    # Add full-record-only fields
    
    # Short description (abstract-like)
    parsed['short_description'] = work_data.get('short-description', '') or ''
    
    # Citation info
    citation_obj = work_data.get('citation', {}) or {}
    parsed['citation_type'] = citation_obj.get('citation-type', '')
    parsed['citation_value'] = citation_obj.get('citation-value', '')
    
    # URL
    url_obj = work_data.get('url', {})
    parsed['url'] = url_obj.get('value', '') if url_obj else ''
    
    # Contributors
    parsed['contributors'] = parse_contributors(work_data.get('contributors'))
    parsed['contributor_count'] = len(parsed['contributors'])
    parsed['contributor_names'] = '; '.join([c['name'] for c in parsed['contributors'] if c['name']])
    parsed['contributor_orcids'] = '; '.join([c['orcid'] for c in parsed['contributors'] if c['orcid']])
    
    # Language
    parsed['language_code'] = work_data.get('language-code', '')
    
    # Country
    parsed['country'] = work_data.get('country', {}).get('value', '') if work_data.get('country') else ''
    
    return parsed


def flatten_external_ids(external_ids: Dict[str, List[Dict]]) -> Dict[str, str]:
    """
    Flatten external IDs dict for CSV export.
    """
    flat = {}
    for id_type, id_list in external_ids.items():
        # Get first 'self' relationship or first value
        values = []
        for item in id_list:
            if item.get('relationship') == 'self':
                values.insert(0, item.get('value', ''))
            else:
                values.append(item.get('value', ''))
        flat[f'id_{id_type}'] = '; '.join(values)
    return flat


print("✅ Parsing functions loaded.")

## Main Query Function

In [ ]:
def query_orcid_works(orcid: str, fetch_full: bool = True, progress_callback=None) -> Tuple[List[Dict], Dict]:
    """
    Query ORCID for all works in a researcher's profile.
    
    Args:
        orcid: ORCID identifier
        fetch_full: If True, fetch full details for each work (slower but more complete)
        progress_callback: Optional function(current, total) for progress updates
    
    Returns:
        (list of work dicts, stats dict)
    """
    orcid = clean_orcid(orcid)
    if not orcid:
        return [], {'error': 'Invalid ORCID format'}
    
    stats = {
        'orcid': orcid,
        'start_time': datetime.now().isoformat(),
        'work_groups': 0,
        'total_works': 0,
        'works_retrieved': 0,
        'errors': []
    }
    
    # Get person info
    person = fetch_person_info(orcid)
    if person:
        stats['person_name'] = f"{person.get('given_name', '')} {person.get('family_name', '')}".strip()
        stats['credit_name'] = person.get('credit_name', '')
    
    # Fetch works summary
    print(f"Fetching works for ORCID {orcid}...")
    groups, _ = fetch_orcid_works_summary(orcid)
    
    if groups is None:
        stats['error'] = 'Failed to fetch works'
        return [], stats
    
    stats['work_groups'] = len(groups)
    
    # Count total works
    total_works = 0
    for group in groups:
        total_works += len(group.get('work-summary', []))
    stats['total_works'] = total_works
    
    print(f"  Found {len(groups)} work groups ({total_works} total works)")
    
    works = []
    work_idx = 0
    
    for group in groups:
        # Each group can have multiple versions of the same work (from different sources)
        # We'll take the preferred (first) one and note the sources
        summaries = group.get('work-summary', [])
        
        if not summaries:
            continue
        
        # Get the preferred summary (first one, usually most complete)
        preferred = summaries[0]
        put_code = preferred.get('put-code')
        
        if fetch_full and put_code:
            # Fetch full details
            time.sleep(config.REQUEST_DELAY)
            full_work = fetch_work_details(orcid, put_code)
            
            if full_work:
                work = parse_work_full(full_work)
            else:
                work = parse_work_summary(preferred)
                stats['errors'].append(f"Could not fetch full details for put-code {put_code}")
        else:
            work = parse_work_summary(preferred)
        
        # Add metadata about duplicates/sources
        work['source_count'] = len(summaries)
        work['all_sources'] = '; '.join([
            s.get('source', {}).get('source-name', {}).get('value', '') 
            if s.get('source', {}).get('source-name') else ''
            for s in summaries
        ])
        
        # Add queried ORCID
        work['queried_orcid'] = orcid
        
        works.append(work)
        work_idx += 1
        
        if progress_callback:
            progress_callback(work_idx, len(groups))
        
        if work_idx % 20 == 0:
            print(f"  Processed {work_idx}/{len(groups)} works...")
    
    stats['works_retrieved'] = len(works)
    stats['end_time'] = datetime.now().isoformat()
    
    print(f"  Retrieved {len(works)} works")
    
    return works, stats


def query_multiple_orcids(orcids: List[str], fetch_full: bool = True, progress_callback=None) -> Tuple[List[Dict], Dict]:
    """
    Query works for multiple ORCIDs.
    """
    all_works = []
    combined_stats = {
        'total_orcids': len(orcids),
        'successful': 0,
        'failed': 0,
        'total_works': 0,
        'per_orcid': [],
        'start_time': datetime.now().isoformat()
    }
    
    for idx, orcid in enumerate(orcids):
        print(f"\n[{idx+1}/{len(orcids)}] Processing {orcid}")
        
        works, stats = query_orcid_works(orcid, fetch_full)
        
        if works:
            all_works.extend(works)
            combined_stats['successful'] += 1
            combined_stats['total_works'] += len(works)
        else:
            combined_stats['failed'] += 1
        
        combined_stats['per_orcid'].append(stats)
        
        if progress_callback:
            progress_callback(idx + 1, len(orcids))
        
        # Delay between ORCIDs
        if idx < len(orcids) - 1:
            time.sleep(config.REQUEST_DELAY)
    
    combined_stats['end_time'] = datetime.now().isoformat()
    
    return all_works, combined_stats


print("✅ Query functions loaded.")

## Test API Connection

In [ ]:
# Test ORCID API with a known researcher
print("Testing ORCID API connection...")
print()

# Tim Ingold - prominent anthropologist
test_orcid = "0000-0002-6243-6224"
print(f"Testing with ORCID: {test_orcid}")

# Quick test - just get works summary, don't fetch full details
groups, info = fetch_orcid_works_summary(test_orcid)

if groups is not None:
    print()
    print("✅ API connection successful!")
    print(f"   Work groups found: {len(groups)}")
    
    if groups:
        # Show first work as sample
        first_group = groups[0]
        first_summary = first_group.get('work-summary', [{}])[0]
        parsed = parse_work_summary(first_summary)
        
        print()
        print("   Sample work:")
        print(f"   Title: {parsed['title'][:60]}..." if len(parsed['title']) > 60 else f"   Title: {parsed['title']}")
        print(f"   Type: {parsed['type']}")
        print(f"   Year: {parsed['publication_year']}")
        print(f"   DOI: {parsed['doi'] or 'N/A'}")
        print(f"   Source: {parsed['source']}")
else:
    print("❌ API test failed")

## Query Interface

In [ ]:
# Global results storage
query_results = []
query_stats = {}

# Interface widgets
query_mode = widgets.RadioButtons(
    options=['Single ORCID', 'Multiple ORCIDs'],
    value='Single ORCID',
    description='Mode:',
    style={'description_width': '80px'}
)

orcid_input = widgets.Text(
    placeholder='0000-0002-1825-0097',
    description='ORCID:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='400px')
)

orcid_textarea = widgets.Textarea(
    placeholder='Enter ORCIDs (one per line):\n0000-0002-1825-0097\n0000-0001-2345-6789',
    description='ORCIDs:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='500px', height='120px')
)

orcid_upload = widgets.FileUpload(
    accept='.csv,.txt',
    multiple=False,
    description='Or upload file'
)

fetch_full_checkbox = widgets.Checkbox(
    value=True,
    description='Fetch full work details (slower but more complete)',
    indent=False,
    layout=widgets.Layout(width='400px')
)

run_button = widgets.Button(
    description='Query ORCID Works',
    button_style='primary',
    icon='search'
)

progress = widgets.IntProgress(
    value=0,
    min=0,
    max=100,
    description='Progress:',
    bar_style='info'
)

output = widgets.Output()

# Visibility toggle
single_box = widgets.VBox([orcid_input])
multi_box = widgets.VBox([orcid_textarea, widgets.HTML('<b>Or upload CSV/TXT with ORCIDs:</b>'), orcid_upload])
multi_box.layout.display = 'none'

def on_mode_change(change):
    if change['new'] == 'Single ORCID':
        single_box.layout.display = 'block'
        multi_box.layout.display = 'none'
    else:
        single_box.layout.display = 'none'
        multi_box.layout.display = 'block'

query_mode.observe(on_mode_change, names='value')

def update_progress(current, total):
    progress.max = total
    progress.value = current

def run_query(button):
    global query_results, query_stats
    query_results = []
    query_stats = {}
    
    with output:
        clear_output()
        progress.value = 0
        
        fetch_full = fetch_full_checkbox.value
        
        if query_mode.value == 'Single ORCID':
            orcid = orcid_input.value.strip()
            if not orcid:
                print("⚠️ Please enter an ORCID.")
                return
            
            orcid = clean_orcid(orcid)
            if not orcid:
                print("⚠️ Invalid ORCID format.")
                return
            
            print(f"🔍 Querying ORCID works for: {orcid}")
            print("=" * 60)
            print()
            
            query_results, query_stats = query_orcid_works(orcid, fetch_full, update_progress)
            
        else:  # Multiple ORCIDs
            orcids = []
            
            # Check for uploaded file
            if orcid_upload.value:
                file_info = list(orcid_upload.value.values())[0]
                content = file_info['content'].decode('utf-8')
                
                if file_info['metadata']['name'].endswith('.csv'):
                    df = pd.read_csv(StringIO(content))
                    # Find ORCID column
                    orcid_col = None
                    for col in df.columns:
                        if 'orcid' in col.lower():
                            orcid_col = col
                            break
                    if orcid_col:
                        raw_orcids = df[orcid_col].dropna().tolist()
                    else:
                        raw_orcids = df.iloc[:, 0].dropna().tolist()
                else:
                    raw_orcids = [line.strip() for line in content.split('\n') if line.strip()]
                
                orcids = [clean_orcid(o) for o in raw_orcids if clean_orcid(o)]
            else:
                raw_orcids = [line.strip() for line in orcid_textarea.value.split('\n') if line.strip()]
                orcids = [clean_orcid(o) for o in raw_orcids if clean_orcid(o)]
            
            if not orcids:
                print("⚠️ No valid ORCIDs found.")
                return
            
            print(f"🔍 Querying ORCID works for {len(orcids)} researchers")
            print("=" * 60)
            
            query_results, query_stats = query_multiple_orcids(orcids, fetch_full, update_progress)
        
        # Display summary
        print()
        print("=" * 60)
        print("📊 QUERY SUMMARY")
        print("=" * 60)
        print(f"Works retrieved: {len(query_results)}")
        
        if query_results:
            # By type
            types = {}
            for work in query_results:
                t = work.get('type', 'unknown')
                types[t] = types.get(t, 0) + 1
            
            print(f"\nBy type:")
            for t, count in sorted(types.items(), key=lambda x: -x[1])[:10]:
                print(f"  • {t}: {count}")
            
            # By year
            years = [w.get('publication_year') for w in query_results if w.get('publication_year')]
            if years:
                print(f"\nYear range: {min(years)} - {max(years)}")
            
            # DOI coverage
            with_doi = sum(1 for w in query_results if w.get('doi'))
            print(f"\nWorks with DOI: {with_doi}/{len(query_results)} ({100*with_doi/len(query_results):.1f}%)")
            
            # Sources
            sources = {}
            for work in query_results:
                s = work.get('source', 'unknown')
                sources[s] = sources.get(s, 0) + 1
            
            print(f"\nTop sources:")
            for s, count in sorted(sources.items(), key=lambda x: -x[1])[:5]:
                print(f"  • {s}: {count}")
        
        print()
        print("✅ Query complete! Run the export cell to download results.")

run_button.on_click(run_query)

# Display interface
display(widgets.HTML('<h3>🔎 ORCID Works Query</h3>'))
display(query_mode)
display(single_box)
display(multi_box)
display(fetch_full_checkbox)
display(widgets.HBox([run_button]))
display(progress)
display(output)

## Preview Results

In [ ]:
# Preview query results
if query_results:
    print(f"📋 Preview of {len(query_results)} works")
    print("=" * 80)
    print()
    
    # Create preview dataframe
    preview_cols = ['queried_orcid', 'title', 'type', 'journal_title', 'publication_year', 'doi', 'source']
    preview_data = []
    for w in query_results:
        row = {col: w.get(col, '') for col in preview_cols}
        preview_data.append(row)
    
    preview_df = pd.DataFrame(preview_data)
    
    # Truncate for display
    preview_df['title'] = preview_df['title'].str[:50] + '...'
    preview_df['journal_title'] = preview_df['journal_title'].str[:30]
    
    display(preview_df.head(25))
    
    if len(query_results) > 25:
        print(f"\n... and {len(query_results) - 25} more works")
else:
    print("⚠️ No results to preview. Run the query cell first.")

## Export Results

In [ ]:
def export_results(works: List[Dict], stats: Dict) -> Tuple[str, str, str]:
    """
    Export query results to CSV and JSON files.
    """
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    # Determine filename base
    if stats.get('orcid'):
        orcid_slug = stats['orcid'].replace('-', '')
        base_name = f"orcid_works_{orcid_slug}"
    else:
        base_name = f"orcid_works_batch_{stats.get('successful', 0)}authors"
    
    # --- CSV Export (flattened) ---
    csv_rows = []
    for work in works:
        row = {
            'queried_orcid': work.get('queried_orcid', ''),
            'put_code': work.get('put_code', ''),
            'title': work.get('title', ''),
            'subtitle': work.get('subtitle', ''),
            'translated_title': work.get('translated_title', ''),
            'type': work.get('type', ''),
            'journal_title': work.get('journal_title', ''),
            'publication_date': work.get('publication_date', ''),
            'publication_year': work.get('publication_year', ''),
            'publication_month': work.get('publication_month', ''),
            'publication_day': work.get('publication_day', ''),
            'doi': work.get('doi', ''),
            'url': work.get('url', ''),
            'short_description': work.get('short_description', ''),
            'contributor_count': work.get('contributor_count', ''),
            'contributor_names': work.get('contributor_names', ''),
            'contributor_orcids': work.get('contributor_orcids', ''),
            'language_code': work.get('language_code', ''),
            'country': work.get('country', ''),
            'source': work.get('source', ''),
            'source_count': work.get('source_count', ''),
            'all_sources': work.get('all_sources', ''),
            'visibility': work.get('visibility', ''),
            'citation_type': work.get('citation_type', ''),
        }
        
        # Flatten external IDs
        ext_ids = work.get('external_ids', {})
        flat_ids = flatten_external_ids(ext_ids)
        row.update(flat_ids)
        
        csv_rows.append(row)
    
    df = pd.DataFrame(csv_rows)
    
    csv_filename = f"{base_name}_{timestamp}.csv"
    csv_path = os.path.join(config.OUTPUT_PATH, csv_filename)
    df.to_csv(csv_path, index=False, encoding='utf-8-sig')
    
    print(f"✅ CSV exported: {csv_filename}")
    print(f"   Rows: {len(df)}")
    print(f"   Columns: {len(df.columns)}")
    
    # --- JSON Export (full nested structure) ---
    json_data = {
        'metadata': {
            'query_stats': stats,
            'export_timestamp': timestamp,
            'work_count': len(works)
        },
        'works': works
    }
    
    json_filename = f"{base_name}_{timestamp}.json"
    json_path = os.path.join(config.OUTPUT_PATH, json_filename)
    
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(json_data, f, indent=2, ensure_ascii=False, default=str)
    
    print(f"\n✅ JSON exported: {json_filename}")
    print(f"   Includes full contributor details and all external IDs")
    
    # --- Stats Export ---
    stats_filename = f"{base_name}_{timestamp}_stats.json"
    stats_path = os.path.join(config.OUTPUT_PATH, stats_filename)
    
    with open(stats_path, 'w', encoding='utf-8') as f:
        json.dump(stats, f, indent=2, default=str)
    
    print(f"\n✅ Stats exported: {stats_filename}")
    
    return csv_path, json_path, stats_path


# Execute export
if query_results:
    print("💾 Exporting ORCID Works")
    print("=" * 60)
    print()
    
    csv_path, json_path, stats_path = export_results(query_results, query_stats)
    
    print()
    print("=" * 60)
    print("🎉 Export complete!")
else:
    print("⚠️ No results to export. Run the query cell first.")

## Download Files

In [ ]:
# Download exported files
def create_download_interface():
    """Create interface for downloading exported files."""
    
    if not os.path.exists(config.OUTPUT_PATH):
        print("⚠️ No output directory found. Run the export cell first.")
        return
    
    files_list = os.listdir(config.OUTPUT_PATH)
    if not files_list:
        print("⚠️ No exported files found. Run the export cell first.")
        return
    
    print("📥 Available Files for Download")
    print("=" * 60)
    print()
    
    # Group files
    csv_files = sorted([f for f in files_list if f.endswith('.csv')])
    json_files = sorted([f for f in files_list if f.endswith('.json') and not f.endswith('_stats.json')])
    stats_files = sorted([f for f in files_list if f.endswith('_stats.json')])
    
    all_files = csv_files + json_files + stats_files
    
    for f in all_files:
        file_path = os.path.join(config.OUTPUT_PATH, f)
        size_kb = os.path.getsize(file_path) / 1024
        
        if size_kb > 1024:
            size_str = f"{size_kb/1024:.1f} MB"
        else:
            size_str = f"{size_kb:.1f} KB"
        
        icon = "📊" if f.endswith('.csv') else "📋" if f.endswith('_stats.json') else "🗂️"
        print(f"{icon} {f} ({size_str})")
    
    print()
    
    if IN_COLAB:
        print("Click buttons to download:")
        print()
        
        for f in all_files:
            file_path = os.path.join(config.OUTPUT_PATH, f)
            
            button = widgets.Button(
                description=f'📥 {f[:45]}...' if len(f) > 45 else f'📥 {f}',
                tooltip=f'Download {f}',
                layout=widgets.Layout(width='450px', height='35px', margin='3px'),
                style={'button_color': '#6096BA'}
            )
            
            def make_handler(path):
                def handler(b):
                    files.download(path)
                return handler
            
            button.on_click(make_handler(file_path))
            display(button)
    else:
        print(f"Files saved to: {config.OUTPUT_PATH}")

create_download_interface()

## Wikidata-Ready Export (Optional)

*Export works in format compatible with AAA_Wikidata_Article_Importer.*

In [ ]:
def export_for_wikidata(works: List[Dict]) -> str:
    """
    Export works in format compatible with Wikidata Article Importer.
    """
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    rows = []
    for w in works:
        # Build author string
        contributors = w.get('contributors', [])
        if contributors:
            authors = '; '.join([c['name'] for c in contributors if c.get('name')])
        else:
            authors = ''
        
        row = {
            'DOI': w.get('doi', ''),
            'Title': w.get('title', ''),
            'Authors': authors,
            'Publication Date': w.get('publication_date', ''),
            'Year': w.get('publication_year', ''),
            'Journal': w.get('journal_title', ''),
            'ISSN': '',  # Not typically in ORCID
            'Volume': '',  # Not typically in ORCID
            'Issue': '',
            'Page': '',
            'URL': w.get('url', ''),
            'Type': w.get('type', ''),
            'ORCID_Source': w.get('queried_orcid', ''),
            'ORCID_PutCode': w.get('put_code', '')
        }
        rows.append(row)
    
    df = pd.DataFrame(rows)
    
    # Filter to only journal articles with DOIs for Wikidata import
    df_importable = df[(df['DOI'] != '') & (df['Type'].isin(['journal-article', 'JOURNAL_ARTICLE']))].copy()
    
    filename = f"orcid_works_for_wikidata_{timestamp}.csv"
    filepath = os.path.join(config.OUTPUT_PATH, filename)
    df_importable.to_csv(filepath, index=False, encoding='utf-8-sig')
    
    print(f"✅ Wikidata-ready CSV exported: {filename}")
    print(f"   Total works: {len(df)}")
    print(f"   Journal articles with DOI (importable): {len(df_importable)}")
    print(f"   Works without DOI (skipped): {(df['DOI'] == '').sum()}")
    print(f"   Non-article types (skipped): {len(df) - len(df_importable) - (df['DOI'] == '').sum()}")
    
    return filepath


if query_results:
    print("🔗 Exporting for Wikidata Import")
    print("=" * 60)
    print()
    
    wikidata_path = export_for_wikidata(query_results)
    
    if IN_COLAB:
        print()
        download_btn = widgets.Button(
            description='📥 Download Wikidata CSV',
            button_style='success',
            layout=widgets.Layout(width='300px')
        )
        
        def download_wikidata(b):
            files.download(wikidata_path)
        
        download_btn.on_click(download_wikidata)
        display(download_btn)
else:
    print("⚠️ No results to export. Run the query cell first.")

## Analysis (Optional)

In [ ]:
# Additional analysis of ORCID works
if query_results:
    print("📈 ORCID Works Analysis")
    print("=" * 60)
    print()
    
    # Publications by year
    years = [w.get('publication_year') for w in query_results if w.get('publication_year')]
    if years:
        year_counts = pd.Series(years).value_counts().sort_index()
        
        print("📅 Publications by Year")
        print("-" * 40)
        for year, count in year_counts.items():
            bar = "█" * min(count, 40)
            print(f"{year}: {bar} {count}")
        print()
    
    # Work types
    types = [w.get('type', 'unknown') for w in query_results]
    type_counts = pd.Series(types).value_counts()
    
    print("📚 Work Types")
    print("-" * 40)
    for work_type, count in type_counts.items():
        print(f"  {count:4d} | {work_type}")
    print()
    
    # External identifier coverage
    id_types = {}
    for w in query_results:
        for id_type in w.get('external_ids', {}).keys():
            id_types[id_type] = id_types.get(id_type, 0) + 1
    
    if id_types:
        print("🆔 External Identifier Coverage")
        print("-" * 40)
        for id_type, count in sorted(id_types.items(), key=lambda x: -x[1]):
            pct = 100 * count / len(query_results)
            print(f"  {id_type:20s}: {count:4d} ({pct:5.1f}%)")
        print()
    
    # Top journals/venues
    journals = [w.get('journal_title') for w in query_results if w.get('journal_title')]
    if journals:
        journal_counts = pd.Series(journals).value_counts().head(15)
        
        print("📰 Top 15 Journals/Venues")
        print("-" * 40)
        for journal, count in journal_counts.items():
            print(f"  {count:3d} | {journal[:50]}")
        print()
    
    # Data sources
    sources = [w.get('source', 'unknown') for w in query_results]
    source_counts = pd.Series(sources).value_counts()
    
    print("📡 Data Sources")
    print("-" * 40)
    for source, count in source_counts.items():
        pct = 100 * count / len(query_results)
        print(f"  {source:30s}: {count:4d} ({pct:5.1f}%)")

else:
    print("⚠️ No results to analyze. Run the query cell first.")

## Compare with CrossRef (Optional)

*If you've also run the CrossRef Author Works Query, you can compare coverage.*

In [ ]:
# Compare ORCID works with CrossRef (manual comparison helper)
if query_results:
    print("🔄 ORCID vs CrossRef Comparison Helper")
    print("=" * 60)
    print()
    
    # Extract DOIs from ORCID results
    orcid_dois = set()
    for w in query_results:
        doi = w.get('doi', '').strip().lower()
        if doi:
            orcid_dois.add(doi)
    
    print(f"DOIs found in ORCID profile: {len(orcid_dois)}")
    print(f"Works without DOI: {len(query_results) - len(orcid_dois)}")
    print()
    
    # Export DOI list for CrossRef comparison
    if orcid_dois:
        doi_filename = f"orcid_dois_for_crossref_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
        doi_path = os.path.join(config.OUTPUT_PATH, doi_filename)
        
        with open(doi_path, 'w') as f:
            for doi in sorted(orcid_dois):
                f.write(doi + '\n')
        
        print(f"📄 DOI list exported: {doi_filename}")
        print("   Use this file with CrossRef_Author_Works_Query.ipynb")
        print("   to compare ORCID profile with CrossRef metadata.")
        
        if IN_COLAB:
            print()
            btn = widgets.Button(
                description='📥 Download DOI List',
                button_style='info',
                layout=widgets.Layout(width='250px')
            )
            def dl(b):
                files.download(doi_path)
            btn.on_click(dl)
            display(btn)
else:
    print("⚠️ No results to compare. Run the query cell first.")